# Day 14: UDFs and Pandas UDFs

Extending Spark with your own Python logic

By now you’ve seen that Spark has a rich set of built-in functions for numbers, strings, arrays, and dates. Most of the time, those functions are all you need. But sooner or later, you’ll run into a case where Spark doesn’t have the exact function you need.

Maybe your business wants a custom rule, like:

- Classify revenue as “High”, “Medium”, or “Low” based on thresholds that don’t map neatly to SQL CASE.
- Clean a messy free-text column using a rule Spark doesn’t natively support.
- Apply a machine learning model prediction inside a Spark pipeline.

In these cases, you reach for UDFs (User-Defined Functions).

### The Idea Behind UDFs
A UDF is your way of teaching Spark a new trick. You write a Python function, then register it with Spark so you can use it in DataFrame transformations just like a built-in function.

But there’s a catch: Spark is distributed, and Python runs on the driver. When you write a Python UDF, Spark has to ship your function to each worker, serialize data into Python objects, run the function row by row, then bring results back into Spark’s JVM world. This serialization/deserialization dance is what makes regular UDFs slower.

### A Simple UDF Example
Let’s return to our sales + feedback dataset from Day 13. Suppose we want to classify each revenue into “High” (>= 1000) or “Low” (< 1000).

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

df = spark.read.csv("/Volumes/thedataengineering_00/data/sampledata/12_sales_csv.csv", header=True, inferSchema=True)

# Step 1: Write a plain Python function
def classify_revenue(amount):
    if amount is None:
        return "Unknown"
    return "High" if amount >= 1000 else "Low"

# Step 2: Register it as a Spark UDF
classify_udf = udf(classify_revenue, StringType())

# Step 3: Use it in DataFrame API
df_classified = df.withColumn("revenue_category", classify_udf(df["revenue"]))
df_classified.show()

Here, Spark called our Python function for each row. That’s the power of UDFs.

### The Performance Trade-off
This works well for small to medium datasets, but if you try to run this on billions of rows, you’ll notice it’s much slower than using Spark’s native functions. Why?

Because native functions are executed inside Spark’s optimized engine (JVM, Tungsten, Catalyst). UDFs force Spark to leave that optimized world, cross into Python, process one row at a time, then return.

So the rule of thumb is:

- Prefer built-in functions whenever possible.
- Use UDFs only when you really need custom logic.

### Pandas UDFs — A Faster Alternative
To solve the performance bottleneck, Spark introduced Pandas UDFs (also called vectorized UDFs). Instead of processing one row at a time, Pandas UDFs process data in batches using Apache Arrow to transfer data efficiently between Spark and Python.

This means your Python function receives a Pandas Series (a whole column chunk), not a single row, and returns another Pandas Series.

### A Pandas UDF Example
Let’s rewrite our revenue classifier as a Pandas UDF.

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Step 1: Write a Pandas function (works on a Pandas Series)
def classify_revenue_batch(amounts: pd.Series) -> pd.Series:
    return amounts.fillna(-1).apply(lambda x: "Unknown" if x == -1 else ("High" if x >= 1000 else "Low"))

# Step 2: Register as Pandas UDF
classify_pandas_udf = pandas_udf(classify_revenue_batch, returnType=StringType())

# Step 3: Use in Spark
df_classified_pandas = df.withColumn("revenue_category", classify_pandas_udf(df["revenue"]))
df_classified_pandas.show()

This looks almost identical from the user’s perspective, but under the hood it’s much faster because Spark ships data in vectorized batches and avoids row-by-row Python calls.

When to Use UDFs vs Pandas UDFs
- Use built-in Spark functions first. They are the fastest and most optimized.
- If something truly cannot be expressed with built-ins, prefer Pandas UDFs for performance.
- Reserve regular UDFs for simple cases or when vectorization isn’t possible.

### A Real-Life Example
Imagine you’re working on an e-commerce pipeline. The marketing team wants to tag each order with a sentiment score based on customer feedback. Spark doesn’t have a “sentiment analysis” function built in.

You might import a Python sentiment library and wrap it inside a Pandas UDF:

In [0]:
%pip install textblob

In [0]:
# Install dependency if needed
%pip install textblob

from textblob import TextBlob
from pyspark.sql.types import FloatType
from pyspark.sql.functions import pandas_udf
import pandas as pd

# Sample data
data = [
    ("I love this product!",),
    ("This is terrible.",),
    ("It's okay, not great.",),
    ("Fantastic experience!",)
]

# Create DataFrame with the 'feedback' column
df = spark.createDataFrame(data, ["feedback"])

# Define UDF
def sentiment_score(feedbacks: pd.Series) -> pd.Series:
    return feedbacks.fillna("").apply(lambda x: TextBlob(x).sentiment.polarity)

sentiment_udf = pandas_udf(sentiment_score, FloatType())

# Apply UDF
df_sentiment = df.withColumn("sentiment", sentiment_udf("feedback"))

df_sentiment.show(truncate=False)


Now each row has a numeric sentiment score between -1 (negative) and +1 (positive). This is the kind of extension UDFs unlock.

Wrapping Up
Today, you learned how to teach Spark new tricks:

UDFs let you wrap ordinary Python functions into Spark transformations, but they can be slow.

Pandas UDFs batch operations using Apache Arrow, making them far faster and more scalable.

Always check whether Spark’s built-in functions already solve your problem — if they do, use them. If not, UDFs are your escape hatch.

As you grow into a Spark data engineer, you’ll often find yourself balancing between flexibility (custom Python) and performance (Spark’s engine). UDFs and Pandas UDFs sit right at that balance point.

In [0]:
Now each row has a numeric sentiment score between -1 (negative) and +1 (positive). This is the kind of extension UDFs unlock.

Wrapping Up
Today, you learned how to teach Spark new tricks:

UDFs let you wrap ordinary Python functions into Spark transformations, but they can be slow.

Pandas UDFs batch operations using Apache Arrow, making them far faster and more scalable.

Always check whether Spark’s built-in functions already solve your problem — if they do, use them. If not, UDFs are your escape hatch.

As you grow into a Spark data engineer, you’ll often find yourself balancing between flexibility (custom Python) and performance (Spark’s engine). UDFs and Pandas UDFs sit right at that balance point.